In [43]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap
from tigramite import plotting as tp

import pcmci_sweep as sw

%load_ext autoreload
%autoreload 2


POS, NEG =  "#eb6834", "#2a78d6"


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [44]:
RUN, WANT_H = "2_pos_ratio_10", "3.5h"

rows = []
for p in sorted(Path("runs_simple").glob("*.npz")):
    c = json.loads(str(np.load(p, allow_pickle=False)["config"]))
    rows.append(dict(name=c["name"], horizon=c.get("HORIZON", "?"),
                     freq=c["FREQ"], tau=c["TAU_MAX"],
                     nvars=len(c["vars"]), file=p))
idx = pd.DataFrame(rows).sort_values(["name", "horizon"])
print(idx.drop(columns="file").to_string(index=False))

# sel = idx.query("name == @RUN and horizon == @WANT_H")
# assert len(sel) == 1, f"expected 1 match for {RUN}/{WANT_H}, got {len(sel)}"
# f = sel["file"].iloc[0]

z   = np.load(f, allow_pickle=False)
cfg = json.loads(str(z["config"]))
var_names      = [str(s) for s in z["var_names"]]
g, v, q_matrix = z["graph"], z["val"], z["q"]

N       = len(var_names)
TAU_MAX = cfg["TAU_MAX"]
FREQ    = cfg["FREQ"]
MINS    = int(pd.Timedelta(FREQ) / pd.Timedelta("1min"))
ALPHA   = cfg["PC_ALPHA"]
j       = var_names.index(sw.TARGET_CLEAN)
print(f"\nloaded {f.name} | {N} vars | tau={TAU_MAX} | {FREQ}")


           name horizon  freq  tau  nvars
 10_main_oxy_15    3.5h 15min   14      6
 11_main_oxy_20    3.5h 20min   10      6
 12_main_oxy_30    3.5h 30min    7      6
  3_main_oxy_10    3.5h 10min   21      6
4_main_ratio_10    3.5h 10min   21      6
 6_main_ratio_5    3.5h  5min   42      6
8_main_ratio_15    3.5h 15min   14      6
   9_main_oxy_5    3.5h  5min   42      6

loaded 2_pos_ratio_10_3.5h_86ebf4ad.npz | 16 vars | tau=21 | 10min


In [45]:
for label, idx in (("causes of ΔNOx", lambda i, t: (i, j, t)),
                   ("effects of ΔNOx", lambda i, t: (j, i, t))):
    rows = [(var_names[i], t, v[idx(i, t)], q_matrix[idx(i, t)])
            for i in range(N) for t in range(TAU_MAX + 1)
            if g[idx(i, t)] == '-->' and i != j and q_matrix[idx(i, t)] < ALPHA]
    print(f"\n--- {label} (q < {ALPHA}) ---")
    if not rows:
        print("  (none)")
    for nm, t, val, qq in sorted(rows, key=lambda r: -abs(r[2])):
        print(f"  {nm:<22} lag {t:>3} ({t*MINS:>4} min)  MCI {val:+.3f}  q {qq:.1e}")

print("\n--- NOx autodependency ---")
for t in range(1, TAU_MAX + 1):
    if g[j, j, t] != '' and q_matrix[j, j, t] < ALPHA:
        print(f"  lag {t:>3} ({t*MINS:>4} min)  MCI {v[j,j,t]:+.3f}  q {q_matrix[j,j,t]:.1e}")



--- causes of ΔNOx (q < 0.01) ---
  ratio_2                lag   1 (  10 min)  MCI +0.236  q 9.5e-62
  ratio_1                lag   1 (  10 min)  MCI +0.215  q 2.4e-53
  ratio_3                lag   1 (  10 min)  MCI +0.139  q 4.0e-22
  ratio_2                lag   0 (   0 min)  MCI +0.104  q 6.2e-14
  oil_1                  lag   1 (  10 min)  MCI -0.072  q 1.2e-05
  oil_4                  lag   1 (  10 min)  MCI -0.062  q 1.9e-03

--- effects of ΔNOx (q < 0.01) ---
  (none)

--- NOx autodependency ---
  lag   1 (  10 min)  MCI -0.315  q 3.4e-138


In [46]:
cols = {}
for p in sorted(Path("runs_simple").glob("*.npz")):
    z2  = np.load(p, allow_pickle=False)
    c   = json.loads(str(z2["config"]))
    vn  = [str(s) for s in z2["var_names"]]
    gg, vv, qq = z2["graph"], z2["val"], z2["q"]
    jj  = vn.index(sw.TARGET_CLEAN)
    m   = int(pd.Timedelta(c["FREQ"]) / pd.Timedelta("1min"))
    key = f"{c['name']}@{c.get('HORIZON','?')}"          # <- name + horizon
    cols[key] = {
        (vn[i], t * m): round(float(vv[i, jj, t]), 3)
        for i in range(len(vn)) for t in range(c["TAU_MAX"] + 1)
        if gg[i, jj, t] == '-->' and i != jj and qq[i, jj, t] < c["PC_ALPHA"]
    }

cmp = pd.DataFrame(cols).sort_index()
cmp.index.names = ["variable", "lag_min"]
print(cmp.fillna("").to_string())



                   10_main_oxy_15@3.5h 11_main_oxy_20@3.5h 12_main_oxy_30@3.5h 3_main_oxy_10@3.5h 4_main_ratio_10@3.5h 6_main_ratio_5@3.5h 8_main_ratio_15@3.5h 9_main_oxy_5@3.5h
variable   lag_min                                                                                                                                                               
oil_main   0                                                                                                                          0.09                                       
           5                                                                                                                                                               -0.411
           10                                                                              -0.355                                                                                
           15                   -0.327                                                                        

In [47]:
NROWS = 12                       # fixed number of bar slots
FIGSZ = (12, 6)
OUT   = Path("plots/pcmci/testsimple"); OUT.mkdir(exist_ok=True)

# ---- pass 1: collect every run and find the shared scale -----------------
runs, gmax = [], 0.0
for p in sorted(Path("runs_simple").glob("*.npz")):
    z2 = np.load(p, allow_pickle=False)
    c  = json.loads(str(z2["config"]))
    vn = [str(s) for s in z2["var_names"]]
    gg, vv, qq = z2["graph"], z2["val"], z2["q"]
    jj = vn.index(sw.TARGET_CLEAN)
    m  = int(pd.Timedelta(c["FREQ"]) / pd.Timedelta("1min"))
    rows = [(vn[i], t * m, float(vv[i, jj, t]))
            for i in range(len(vn)) for t in range(c["TAU_MAX"] + 1)
            if gg[i, jj, t] == '-->']
    rows.sort(key=lambda r: abs(r[2]))
    rows = rows[-NROWS:]
    runs.append((c, rows))
    if rows:
        gmax = max(gmax, abs(rows[-1][2]))

XLIM = np.ceil(gmax * 20) / 20
print(f"{len(runs)} runs | shared x-limit ±{XLIM}")

NROWS = max((len(r) for _, r in runs), default=1)
FIGSZ = (10, 0.42 * NROWS + 1.6)          # height follows the global row count
print(f"{len(runs)} runs | {NROWS} slots | x-limit ±{XLIM}")


# ---- pass 2: draw and save ----------------------------------------------
for c, rows in runs:
    tag    = f"{c['name']}_{c.get('HORIZON','?')}"
    labels = [f"{n}   ·   {lag} min" for n, lag, _ in rows]
    vals   = [x for _, _, x in rows]

    fig, ax = plt.subplots(figsize=FIGSZ)
    ax.barh(range(len(rows)), vals, height=0.62,
            color=[POS if x >= 0 else NEG for x in vals])
    ax.set_yticks(range(len(rows))); ax.set_yticklabels(labels, fontsize=9)
    ax.set_ylim(-0.7, len(rows) - 0.3)       
    ax.set_xlim(-XLIM, XLIM)                        # shared scale
    ax.axvline(0, color="#999", lw=1)
    ax.set_xlabel("MCI partial correlation")
    ax.set_title(f"What drives ΔNOx — {c['name']}   "
                 f"({c['FREQ']}, horizon {c.get('HORIZON','?')})", loc="left")
    ax.grid(axis="x", color="#e8e8e8", lw=.8); ax.set_axisbelow(True)
    for s in ("top", "right", "left"): ax.spines[s].set_visible(False)
    for k, x in enumerate(vals):
        ax.text(x + XLIM * 0.015 * (1 if x >= 0 else -1), k, f"{x:+.3f}",
                va="center", ha="left" if x >= 0 else "right",
                fontsize=8, color="#444")
    ax.legend(handles=[Patch(color=POS, label="raises NOx"),
                       Patch(color=NEG, label="lowers NOx")],
              frameon=False, loc="lower right", fontsize=9)
    fig.tight_layout()
    fig.savefig(OUT / f"bars_{tag}.png", dpi=200)
    plt.close(fig)
    print(f"saved nox_bars_{tag}.png  ({len(rows)} links)")


8 runs | shared x-limit ±0.45
8 runs | 9 slots | x-limit ±0.45
saved nox_bars_10_main_oxy_15_3.5h.png  (4 links)
saved nox_bars_11_main_oxy_20_3.5h.png  (2 links)
saved nox_bars_12_main_oxy_30_3.5h.png  (2 links)
saved nox_bars_3_main_oxy_10_3.5h.png  (4 links)
saved nox_bars_4_main_ratio_10_3.5h.png  (4 links)
saved nox_bars_6_main_ratio_5_3.5h.png  (8 links)
saved nox_bars_8_main_ratio_15_3.5h.png  (3 links)
saved nox_bars_9_main_oxy_5_3.5h.png  (9 links)


In [48]:
CMAP_N  = 'RdYlBu_r'        # 'coolwarm' | 'RdBu_r' | 'PuOr' | 'BrBG' | NOX_DIV
CMAP_E = 'coolwarm'   
E     = 0.5               # colour range, fixed across every figure
FIGSZ = (36, 20)

LABEL_FS   = 20      # variable names        (was 11)
NODE_SIZE  = 0.25    # node circle radius    (was 0.12)
LINK_FS    = 20      # the lag numbers on edges
ARROW_LW   = 7       # edge thickness


OUT   = Path("plots/pcmci/testsimple"); OUT.mkdir(exist_ok=True)
rank = lambda nm: (0 if nm.startswith(("ARCH", "TEMP", "MELTER", "arch_3_4", "bt")) else
                   1 if nm.endswith(("_1")) else
                   2 if nm.endswith(("_2")) else
                   3 if nm.endswith(("_3")) else
                   4 if nm.endswith(("_4")) else
                   5 if nm.endswith(("_main")) else
                   6 if nm.startswith(("WEATHER")) else
                   7 if nm == sw.TARGET_CLEAN else 8)

def draw(vv, gg, vn, tau, idx_keep, fname, title):
    order = sorted(idx_keep, key=lambda i: (rank(vn[i]), vn[i]))
    sub   = np.ix_(order, order, range(tau + 1))
    ang   = np.linspace(0, 2 * np.pi, len(order), endpoint=False)
    tp.plot_graph(
        val_matrix=vv[sub], graph=gg[sub],
        var_names=[vn[i] for i in order],
        node_pos={'x': np.cos(ang), 'y': np.sin(ang)},
        link_colorbar_label='cross-MCI', node_colorbar_label='auto-MCI',
        cmap_edges=CMAP_E, vmin_edges=-E, vmax_edges=E, edge_ticks=E / 4, 
        cmap_nodes=CMAP_N, vmin_nodes=-E, vmax_nodes=E, node_ticks=E / 4, 
        node_size=NODE_SIZE,
        node_aspect=1.0,              # >1 = wider ellipses, fits long names
        arrow_linewidth=ARROW_LW,
        arrowhead_size=20,
        curved_radius=0.25,
        node_label_size=LABEL_FS,
        link_label_fontsize=LINK_FS,
        label_fontsize=12,
        tick_label_size=12,
        figsize=FIGSZ)
    plt.savefig(OUT / fname, dpi=200, bbox_inches="tight", pad_inches=0.15)

    plt.close()

for p in sorted(Path("runs_simple").glob("*.npz")):
    z2     = np.load(p, allow_pickle=False)
    c      = json.loads(str(z2["config"]))
    vn     = [str(s) for s in z2["var_names"]]
    gg, vv = z2["graph"], z2["val"]
    tau    = c["TAU_MAX"]
    jj     = vn.index(sw.TARGET_CLEAN)
    tag    = f"{c['name']}_{c.get('HORIZON','?')}"
    ttl    = f"{c['name']}  ({c['FREQ']}, horizon {c.get('HORIZON','?')})"

    draw(vv, gg, vn, tau, list(range(len(vn))),
         f"all_{tag}.png", f"all links — {ttl}")

    keep = sorted({jj} | {i for i in range(len(vn)) for t in range(tau + 1)
                          if gg[i, jj, t] != '' or gg[jj, i, t] != ''})
    draw(vv, gg, vn, tau, keep,
         f"nox_{tag}.png", f"NOx neighbourhood — {ttl}")

    print(f"{tag}: all={len(vn)} nodes, nox={len(keep)} nodes")


10_main_oxy_15_3.5h: all=6 nodes, nox=5 nodes
11_main_oxy_20_3.5h: all=6 nodes, nox=4 nodes
12_main_oxy_30_3.5h: all=6 nodes, nox=4 nodes
3_main_oxy_10_3.5h: all=6 nodes, nox=4 nodes
4_main_ratio_10_3.5h: all=6 nodes, nox=3 nodes
6_main_ratio_5_3.5h: all=6 nodes, nox=4 nodes
8_main_ratio_15_3.5h: all=6 nodes, nox=4 nodes
9_main_oxy_5_3.5h: all=6 nodes, nox=4 nodes


In [42]:
from collections import defaultdict

RUNS_DIR = Path("runs_agg_v2")
summary  = []

for p in sorted(RUNS_DIR.glob("*.npz")):
    z2  = np.load(p, allow_pickle=False)
    c   = json.loads(str(z2["config"]))
    vn  = [str(s) for s in z2["var_names"]]
    gg, vv, qq = z2["graph"], z2["val"], z2["q"]
    tau, a = c["TAU_MAX"], c["PC_ALPHA"]
    jj  = vn.index(sw.TARGET_CLEAN)
    m   = int(pd.Timedelta(c["FREQ"]) / pd.Timedelta("1min"))

    print(f"\n{'=' * 78}")
    print(f"{c['name']}   {c['FREQ']}   tau={tau}   horizon={c.get('HORIZON','?')}"
          f"   {len(vn)} vars")
    print("=" * 78)

    for label, idx in (("causes of ΔNOx",  lambda i, t: (i, jj, t)),
                       ("effects of ΔNOx", lambda i, t: (jj, i, t))):
        byvar = defaultdict(list)
        for i in range(len(vn)):
            for t in range(tau + 1):
                if gg[idx(i, t)] == '-->' and i != jj and qq[idx(i, t)] < a:
                    byvar[vn[i]].append((t, float(vv[idx(i, t)])))

        print(f"  --- {label} ---")
        if not byvar:
            print("    (none)")
        for nm, lst in sorted(byvar.items(),
                              key=lambda kv: -max(abs(x) for _, x in kv[1])):
            mixed = len({np.sign(x) for _, x in lst}) > 1
            if mixed:
                summary.append((c["name"], label, nm))
            print(f"    {nm:<16}"
                  + "  ".join(f"{t*m:>4}min:{x:+.3f}" for t, x in sorted(lst))
                  + ("   <-- SIGN CHANGE" if mixed else ""))

print(f"\n{'=' * 78}\nsign changes across all runs: {len(summary)}")
for run, label, nm in summary:
    print(f"  {run:<18} {label:<16} {nm}")





1_pos_oxy_10   10min   tau=21   horizon=3.5h   16 vars
  --- causes of ΔNOx ---
    oil_1             10min:-0.277
    oil_2              0min:-0.051    10min:-0.215
    oxy_1             10min:+0.204
    oxy_2              0min:+0.066    10min:+0.163
    oil_3             10min:-0.150
    oxy_3             10min:+0.109
    ARCH #1          140min:-0.080
    oil_4             10min:-0.073
  --- effects of ΔNOx ---
    (none)

2_pos_ratio_10   10min   tau=21   horizon=3.5h   16 vars
  --- causes of ΔNOx ---
    ratio_2            0min:+0.104    10min:+0.236
    ratio_1           10min:+0.215
    ratio_3           10min:+0.139
    oil_1             10min:-0.072
    oil_4             10min:-0.062
  --- effects of ΔNOx ---
    (none)

3_main_oxy_10   10min   tau=21   horizon=3.5h   10 vars
  --- causes of ΔNOx ---
    oil_main          10min:-0.352
    oxy_main          10min:+0.320
    ARCH #1          140min:-0.094
  --- effects of ΔNOx ---
    oxy_main           0min:+0.220

4_main_rat